# Data preprocessing and EDA

This notebook loads `reviews_badminton/data.csv`, performs initial inspection, converts ratings to binary sentiment labels, and demonstrates text cleaning and basic EDA (rating distribution, common words, and wordclouds).

In [ ]:

import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

DATA_PATH = os.path.join('..', 'reviews_badminton', 'data.csv')
df = pd.read_csv(DATA_PATH)
print('Loaded', df.shape, 'rows and columns')
df.head()

In [ ]:

print('Columns:', df.columns.tolist())
print('
Missing values per column:
', df.isnull().sum())
print('
Ratings distribution:')
print(df['Ratings'].value_counts(dropna=False).sort_index())

In [ ]:

def rating_to_sentiment(r):
    try:
        r = float(r)
    except:
        return None
    if r >= 4:
        return 'positive'
    if r <= 2:
        return 'negative'
    return 'neutral'

df['sentiment'] = df['Ratings'].apply(rating_to_sentiment)
print(df['sentiment'].value_counts(dropna=False))
# Optionally drop neutrals
df = df[df['sentiment'] != 'neutral'].copy()
print('After dropping neutral, shape:', df.shape)

In [ ]:

import unicodedata
import string
from nltk.corpus import stopwords
STOPWORDS = set(stopwords.words('english'))

contraction_map = {:,:,:,:,:,:,:,:,:,:}

def expand_contractions(text):
    for k, v in contraction_map.items():
        text = text.replace(k, v)
    return text

def clean_text(text):
    if not isinstance(text, str):
        return ''
  
    text = re.sub(r'ReAD MORE', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'READ MORE', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'READMORE', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'+', ' ', text)
   
    text = unicodedata.normalize('NFKC', text)
    
    text = re.sub(r
, ' ', text)
    text = text.lower()
    text = expand_contractions(text)
    tokens = [t for t in text.split() if t not in STOPWORDS]
    return ' '.join(tokens)


df['clean_text'] = df['Review text'].fillna('').apply(clean_text)
df['clean_text'].head()

In [ ]:

import spacy
try:
    nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
except OSError:
    print('Please run: python -m spacy download en_core_web_sm')
    nlp = None

def lemmatize_text(text, nlp):
    if not nlp or not text:
        return text
    doc = nlp(text)
    return ' '.join([token.lemma_ for token in doc if token.lemma_ != '-PRON-'])

if nlp:
    df['lemma'] = df['clean_text'].apply(lambda x: lemmatize_text(x, nlp))
    df['lemma'].head()

In [ ]:

plt.figure(figsize=(6,4))
sns.countplot(x='sentiment', data=df)
plt.title('Sentiment distribution')
plt.show()

# Wordcloud for negative reviews
neg_text = ' '.join(df[df['sentiment']=='negative']['clean_text'].astype(str).tolist())
pos_text = ' '.join(df[df['sentiment']=='positive']['clean_text'].astype(str).tolist())

wc = WordCloud(width=800, height=400, background_color='white').generate(neg_text)
plt.figure(figsize=(10,5))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Negative reviews wordcloud')
plt.show()

wc2 = WordCloud(width=800, height=400, background_color='white').generate(pos_text)
plt.figure(figsize=(10,5))
plt.imshow(wc2, interpolation='bilinear')
plt.axis('off')
plt.title('Positive reviews wordcloud')
plt.show()